# YOLO inference on CODaN test images

This notebook loads a trained YOLO classifier weights file and runs inference on the images stored under the CODaN test directory.

# Parameters and imports

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from ultralytics import YOLO

# Support functions

In [16]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while True:
        if (current / 'data' / 'CODaN' / 'test').exists():
            return current
        if current.parent == current:
            raise FileNotFoundError('Could not find the repository root from the current working directory.')
        current = current.parent

# Setup

In [10]:
repo_root = find_repo_root(Path.cwd())
weights_path = repo_root / 'src' / 'models' / 'models' / 'CODaN' / 'yolov8n-cls' / 'train' / 'weights' / 'best.pt'
test_dir = repo_root / 'data' / 'CODaN' / 'test'
output_dir = repo_root / 'src' / 'models' / 'models' / 'CODaN' / 'yolov8n-cls' / 'test_predictions'
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Repo root: {repo_root}')
print(f'Weights file: {weights_path}')
print(f'Test directory: {test_dir}')
print(f'Output directory: {output_dir}')

if not weights_path.exists():
    raise FileNotFoundError(f'Weights file not found: {weights_path}')
if not test_dir.exists():
    raise FileNotFoundError(f'Test directory not found: {test_dir}')

image_paths = sorted(
    [p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}]
)
if not image_paths:
    raise FileNotFoundError(f'No image files found in {test_dir}')

print(f'Found {len(image_paths)} image files for inference')
print( f'Image files: {[p.name for p in image_paths[:2]]}{"..." if len(image_paths) > 2 else ""}' )

Repo root: /home/schelian/prjs/python_project_template
Weights file: /home/schelian/prjs/python_project_template/src/models/models/CODaN/yolov8n-cls/train/weights/best.pt
Test directory: /home/schelian/prjs/python_project_template/data/CODaN/test
Output directory: /home/schelian/prjs/python_project_template/src/models/models/CODaN/yolov8n-cls/test_predictions
Found 20 image files for inference
Image files: ['day_Bicycle_000000171351.jpg', 'day_Boat_000000470308.jpg']...


# Testing

In [25]:
# test
model = YOLO(str(weights_path))
results = model([str(p) for p in image_paths], stream=True)

rows = []
for idx, result in enumerate(results):
    original_image_path = image_paths[idx]
    original_image_name = original_image_path.name
    if ( idx < 2 or idx >= len(image_paths) - 2 ):
        print( f'Image: {original_image_path}' )
    
    predicted_class = result.names[result.probs.top1]
    predicted_class_idx = int( predicted_class ) + 1  # important go from [0,1] to [1,2]; @todo make this more robust to changes in class names
    confidence = float(result.probs.top1conf)
    rows.append({
        'image': original_image_name,
        'predicted_class': predicted_class_idx,
        'confidence': confidence,
    })

predictions_df = pd.DataFrame(rows)
print( f'Predictions DataFrame:\n{predictions_df.head(2)}' )

labels_path = test_dir / 'test_images_and_labels.csv'
if not labels_path.exists():
    raise FileNotFoundError(f'Label file not found: {labels_path}')

labels_df = pd.read_csv(labels_path)
labels_df['image'] = labels_df['filename'].astype(str).apply(lambda x: Path(x).name)
labels_df = labels_df[['image', 'class']].rename(columns={'class': 'true_class'})
print( f'Labels DataFrame:\n{labels_df.head(2)}' )

predictions_df = predictions_df.merge(labels_df, on='image', how='inner')
predictions_df['correct'] = predictions_df['predicted_class'] == predictions_df['true_class']

predictions_csv = output_dir / 'predictions.csv'
predictions_df.to_csv(predictions_csv, index=False)
print(f'Saved predictions to {predictions_csv}')


0: 224x224 1 0.79, 0 0.21, 4.5ms
1: 224x224 0 0.99, 1 0.01, 4.5ms
2: 224x224 0 0.99, 1 0.01, 4.5ms
3: 224x224 1 0.84, 0 0.16, 4.5ms
4: 224x224 0 0.96, 1 0.04, 4.5ms
5: 224x224 0 0.83, 1 0.17, 4.5ms
6: 224x224 0 0.91, 1 0.09, 4.5ms
7: 224x224 0 0.95, 1 0.05, 4.5ms
8: 224x224 1 0.86, 0 0.14, 4.5ms
9: 224x224 0 0.97, 1 0.03, 4.5ms
10: 224x224 1 1.00, 0 0.00, 4.5ms
11: 224x224 1 0.95, 0 0.05, 4.5ms
12: 224x224 1 0.99, 0 0.01, 4.5ms
13: 224x224 1 0.98, 0 0.02, 4.5ms
14: 224x224 1 1.00, 0 0.00, 4.5ms
15: 224x224 1 0.95, 0 0.05, 4.5ms
16: 224x224 1 0.76, 0 0.24, 4.5ms
17: 224x224 0 0.91, 1 0.09, 4.5ms
18: 224x224 1 0.98, 0 0.02, 4.5ms
19: 224x224 1 1.00, 0 0.00, 4.5ms
Image: /home/schelian/prjs/python_project_template/data/CODaN/test/day_Bicycle_000000171351.jpg
Image: /home/schelian/prjs/python_project_template/data/CODaN/test/day_Boat_000000470308.jpg
Image: /home/schelian/prjs/python_project_template/data/CODaN/test/night_Dog_2015_05657.png
Image: /home/schelian/prjs/python_project_templa

In [28]:
# metrics
class_names = ['day', 'night']
labels = [1, 2]
cm = confusion_matrix(predictions_df['true_class'], predictions_df['predicted_class'], labels=labels)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
cm_csv = output_dir / 'confusion_matrix.csv'
cm_df.to_csv(cm_csv)
print(f'Saved confusion matrix to {cm_csv}')
print('\nConfusion matrix:')
print(cm_df)

accuracy = accuracy_score(predictions_df['true_class'], predictions_df['predicted_class'])
print(f'Accuracy: {accuracy:.4f}')

report = classification_report(
    predictions_df['true_class'],
    predictions_df['predicted_class'],
    labels=labels,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
report_csv = output_dir / 'classification_report.csv'
report_df.to_csv(report_csv)
print('\nClassification report:')
print(report_df)
print(f'Saved classification report to {report_csv}')

predictions_df.head()

Saved confusion matrix to /home/schelian/prjs/python_project_template/src/models/models/CODaN/yolov8n-cls/test_predictions/confusion_matrix.csv

Confusion matrix:
       day  night
day      7      3
night    1      9
Accuracy: 0.8000

Classification report:
              precision  recall  f1-score  support
day              0.8750     0.7  0.777778     10.0
night            0.7500     0.9  0.818182     10.0
accuracy         0.8000     0.8  0.800000      0.8
macro avg        0.8125     0.8  0.797980     20.0
weighted avg     0.8125     0.8  0.797980     20.0
Saved classification report to /home/schelian/prjs/python_project_template/src/models/models/CODaN/yolov8n-cls/test_predictions/classification_report.csv


,image,predicted_class,confidence,true_class,correct
0,day_Bicycle_000000171351.jpg,2,0.788151,1,False
1,day_Boat_000000470308.jpg,1,0.993451,1,True
2,day_Bottle_000000298138.jpg,1,0.987837,1,True
3,day_Bus_000000024600.jpg,2,0.840161,1,False
4,day_Car_ILSVRC2012_val_00003779.JPEG,1,0.956162,1,True
